In [19]:
File = [['train-v1.1[1].json', 'dev-v1.1[1].json'], ['coqa-dev-v1.0[2].json', 'coqa-train-v1.0[1].json']]
prompt = ["""You are a student writing the examination on the given topic. Stay to the point and be consistent in the answers you give.
    Give the output in the given form:
    Q1) In what year was the Grotto of Our Lady of Lourdes at Notre Dame constructed?
    Ans:\nPoint to note over here is that you are expected to give an accurate and a precise answer for every question given.""",
    """You are a person who is knowledgeable on many topics, while being on the extroverted side. You tend to be chatty,
    while being on the point. Give the output in the given form:
    Q1) What is the capital of Poland?
    Ans:\nThe unique features which should be followed by you include the following points:
    1.) The questions are conversational.
    2.) The answers can be free-form text.
    3.) Each answer also comes with an evidence subsequence highlighted in the passage.
    4.) The passages are collected from seven diverse domains."""]
questions = ["In what year was the Grotto of Our Lady of Lourdes at Notre Dame constructed?", "What is the capital of Poland?"]

In [20]:
from transformers import pipeline
import json
import os
from warnings import filterwarnings
filterwarnings('ignore')

In [21]:
def LenCheck(File):
  if len(File) == 0:
    print("Sorry, cannot continue over here!!!")
    return 0
  else:
    return 1 #This is the helper function over here!!!

In [22]:
def Part1(File, question):
    # Input validation
    if 0 in [LenCheck(File), LenCheck(question)]:
      return
    count = 0
    # Process each file
    for file_location in File:
        # Check if file exists
        if not os.path.exists(file_location):
            print(f"ERROR: File '{file_location}' does not exist!")
            print(f"Current directory: {os.getcwd()}")
            print(f"Available JSON files: {[f for f in os.listdir('.') if f.endswith('.json')]}")
            continue
        try:
            # Extract context from JSON file
            context = extract_context_from_json(file_location)
            if not context:
                print(f"WARNING: No context extracted from {file_location}")
                continue
            print(f"Successfully extracted context")
            print(f"Context length: {len(context)} characters")
            print(f"Preview: {context[:200]}...")
            # Initialize the QA model
            print("\nLoading QA model (this may take a moment)...")
            Model = pipeline('question-answering', model='google-bert/bert-base-uncased')
            print("Model loaded successfully")
            # Get answer
            if "dev-v1" in file_location:
              print(f"\nGetting answer for question: {question}")
              Output = Model(question=question, context=context)
              # Display results
              print(f"RESULT FOR FILE {count}: {file_location}")
              print(f"Question: {question}")
              print(f"Answer: {Output['answer']}")
              print(f"Confidence: {Output['score']:.3f}")
              print(f"Answer position: {Output['start']} to {Output['end']}")
        except json.JSONDecodeError as e:
            print(f"ERROR: Invalid JSON in {file_location}")
            print(f"Details: {str(e)}")
        except Exception as e:
            print(f"ERROR processing {file_location}: {str(e)}")
            import traceback
            traceback.print_exc()
        count += 1
def extract_context_from_json(file_path):
    print(f"Parsing JSON file...")
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    context_parts = []
    # Check if it's SQuAD format
    if isinstance(data, dict) and 'data' in data:
        print("Detected SQuAD format")
        for article in data['data']:
            title = article.get('title', '')
            if title:
                context_parts.append(f"Title: {title}")
            for paragraph in article.get('paragraphs', []):
                context = paragraph.get('context', '')
                if context:
                    context_parts.append(context)
                # Also extract questions/answers for additional context
                for qa in paragraph.get('qas', []):
                    question = qa.get('question', '')
                    if question:
                        context_parts.append(f"Question: {question}")
                    for answer in qa.get('answers', []):
                        ans_text = answer.get('text', '')
                        if ans_text:
                            context_parts.append(f"Answer: {ans_text}")
    # Check if it's CoQA format
    elif isinstance(data, dict) and 'data' in data and isinstance(data['data'], list):
        print("Detected CoQA format")
        for conversation in data['data']:
            # Add the story/context
            story = conversation.get('story', '')
            if story:
                context_parts.append(story)
            # Add questions and answers for context
            for question_data in conversation.get('questions', []):
                q_text = question_data.get('input_text', '')
                if q_text:
                    context_parts.append(f"Q: {q_text}")
            for answer_data in conversation.get('answers', []):
                a_text = answer_data.get('input_text', '')
                if a_text:
                    context_parts.append(f"A: {a_text}")
    # If it's a simple JSON with text field
    elif isinstance(data, dict) and 'text' in data:
        print("Detected simple text format")
        context_parts.append(data['text'])
    # If it's a list of strings or objects
    elif isinstance(data, list):
        print("Detected list format")
        for item in data:
            if isinstance(item, str):
                context_parts.append(item)
            elif isinstance(item, dict):
                # Try common text fields
                for field in ['text', 'context', 'content', 'story', 'passage']:
                    if field in item and item[field]:
                        context_parts.append(item[field])
                        break
    # Fallback: convert entire JSON to string
    else:
        print("Unknown format, using full JSON string")
        context_parts.append(json.dumps(data))
    # Join all parts with spaces
    full_context = " ".join(context_parts)
    full_context = full_context.strip() #This is to reduce the amount of unnecessary space taken
    # Clean up the text (remove extra whitespace)
    full_context = ' '.join(full_context.split())
    print(f"Extracted {len(context_parts)} text segments")
    return full_context

In [23]:
def check_files_exist(file_lists):
    print("\nChecking for JSON files in current directory...")
    print(f"Current directory: {os.getcwd()}")
    all_files = []
    for file_list in file_lists:
        all_files.extend(file_list)
    existing_files = []
    missing_files = []
    for file_path in all_files:
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path)
            print(f"Found: {file_path} ({file_size:,} bytes)")
            existing_files.append(file_path)
        else:
            print(f"Missing: {file_path}")
            missing_files.append(file_path)
    # Also show any other JSON files in directory
    other_jsons = [f for f in os.listdir('.') if f.endswith('.json') and f not in all_files]
    if other_jsons:
        print(f"\nOther JSON files available:")
        for f in other_jsons:
            print(f"   - {f} ({os.path.getsize(f):,} bytes)")
    return existing_files, missing_files

In [24]:
def Prints():
  print("\nNo files found! Please upload the JSON files to Colab.")
  print("\nTo upload files in Colab, use:")
  print("from google.colab import files")
  print("uploaded = files.upload()")

In [25]:
def main():
    # First, check which files exist
    existing_files, missing_files = check_files_exist(File)
    if not existing_files:
        Prints()
        return
    if len(missing_files) > 0:
        print("\nThe following files are missing:")
        if len(missing_files) == 1:
          Prints()
        else:
          for f in missing_files:
            Prints()
    # Process each question with its corresponding files
    for i in range(len(File)):
        print(f"Prompt: {prompt[i][:100]}...")
        print(f"Question: {questions[i]}")
        print(f"Target files: {File[i]}")
        # Extract just the question part (remove "Q1) " if present)
        clean_question = questions[i].replace("Q1) ", "")
        clean_question = clean_question.strip()
        # Check if question is in prompt (your original condition)
        if questions[i] in prompt[i]:
            print(f"\n✓ Question found in prompt. Processing...")
            # Pass only the files that exist from this group
            available_files = [f for f in File[i] if f in existing_files]
            if available_files:
                Part1(available_files, clean_question)
            else:
                print(f"None of the files in group {i+1} exist!")
                break
        else:
            print(f"\n✗ Question NOT found in prompt!")
            print(f"This condition failed, so Part1 won't be called.")
            break
# Run the program
if __name__ == "__main__":
    main()


Checking for JSON files in current directory...
Current directory: /content
Found: train-v1.1[1].json (30,288,272 bytes)
Found: dev-v1.1[1].json (4,854,279 bytes)
Missing: coqa-dev-v1.0[2].json
Found: coqa-train-v1.0[1].json (49,001,836 bytes)

Other JSON files available:
   - coqa-dev-v1.0[1].json (9,090,845 bytes)

The following files are missing:

No files found! Please upload the JSON files to Colab.

To upload files in Colab, use:
from google.colab import files
uploaded = files.upload()
Prompt: You are a student writing the examination on the given topic. Stay to the point and be consistent in...
Question: In what year was the Grotto of Our Lady of Lourdes at Notre Dame constructed?
Target files: ['train-v1.1[1].json', 'dev-v1.1[1].json']

✓ Question found in prompt. Processing...
Parsing JSON file...
Detected SQuAD format
Extracted 194536 text segments
Successfully extracted context
Context length: 22637643 characters
Preview: Title: University_of_Notre_Dame Architecturally, the

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly init

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully

Getting answer for question: In what year was the Grotto of Our Lady of Lourdes at Notre Dame constructed?


KeyboardInterrupt: 